# Notebook 2 · Mini pipeline

From three passages to about ten screened items, in eight steps:

```
1. write three passages (B1, B2, C1)      2. check their CEFR level, rewrite if off
3. write question + answer candidates      4. rate them (1-10)
5. improve the weak ones and re-rate       6. keep the good ones, pick ~10
7. write distractors, gate, select 3       8. assemble, explain, export JSON
```

Every step is one function with a short prompt.

*Notebook by Rasmus Jensen (ETH Zurich), adapted for the fall school by Rudolf Debelak (EPFL). The pipeline follows Jensen (2025, 2026).*


## Setup

Three things to fill in: the endpoint, the key, and the two model names (one model writes the
passages, one writes and judges the items). On Colab, store the key once under the key icon on
the left (*Secrets*) as `AIG_API_KEY` and the endpoint as `AIG_BASE_URL`; the cell reads them
from there. Everything else in the notebook only uses the three small helpers defined below:

* `render(template, **values)` fills a prompt template — `{{level}}` in the template becomes the
  value of `level`.
* `ask(system, user, schema)` sends one system message and one user message to the model. If a
  `schema` (a pydantic class) is given, the server is asked to reply in that JSON layout and the
  reply comes back as an object of that class, so the rest of the code never parses text.
  Every call is stored in `cache/`; an identical call is replayed from disk, so a notebook that
  has been run once runs again without the endpoint (set `OFFLINE = True` to enforce that).
  Failed calls are retried; a reply that does not fit the schema is asked for once more.
* `ask_many(calls)` runs a list of such calls in parallel, because a pipeline step usually asks
  the same question about many items.


In [ ]:
%pip install -q openai pydantic jinja2 pandas

import json, re, os, time, hashlib
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError
from jinja2 import Template
import pandas as pd

# --- endpoint, key, models -----------------------------------------------------------------
BASE_URL   = os.environ.get("AIG_BASE_URL", "https://YOUR-ENDPOINT/v1")   # the OpenAI-compatible endpoint
API_KEY    = os.environ.get("AIG_API_KEY", "EMPTY")
try:                                                     # on Colab: read both from the Secrets panel
    from google.colab import userdata
    API_KEY  = userdata.get("AIG_API_KEY") or API_KEY
    BASE_URL = userdata.get("AIG_BASE_URL") or BASE_URL
except Exception:
    pass
TEXT_MODEL = "swiss-ai/Apertus-v1.5-70B"              # writes the passages
ITEM_MODEL = "Qwen/Qwen3-30B-A3B-Instruct-2507"       # writes and judges the items; the `reasoning` fields in the schemas carry its chain of thought

# --- cache: every call is stored; identical calls are replayed ------------------------------
CACHE_DIR = Path("cache"); CACHE_DIR.mkdir(exist_ok=True)
OFFLINE   = False                                     # True: never call the endpoint, only replay

client = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=240, max_retries=0)


def render(template, **values):
    return Template(template).render(**values)          # {{name}} in the template -> values[name]


def _parse(text, schema):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)   # thinking models: drop the reasoning block
    text = re.sub(r"```(?:json)?", "", text).strip()              # and code fences
    if schema is None:
        return text
    data = json.loads(text)
    if isinstance(data, list) and len(data) == 1:                # some models wrap the object in a list
        data = data[0]
    return schema.model_validate(data)


def ask(system, user, schema=None, model=None, temperature=0.4, seed=None, think=None, max_tokens=4096, retries=3):
    """One system + one user message -> plain text, or an instance of `schema`."""
    payload = dict(model=model or ITEM_MODEL, temperature=temperature, max_tokens=max_tokens,
                   messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
    if schema is not None:                                        # ask the server for the schema's JSON layout
        payload["response_format"] = {"type": "json_schema",
                                      "json_schema": {"name": schema.__name__, "schema": schema.model_json_schema()}}
    if seed is not None:
        payload["seed"] = seed
    if think is not None:                                         # thinking on/off for Qwen3 hybrid models (vLLM); a no-op for Instruct / Thinking-only models
        payload["extra_body"] = {"chat_template_kwargs": {"enable_thinking": bool(think)}}

    path = CACHE_DIR / (hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:24] + ".json")
    for attempt in range(retries):
        cached = None
        if path.exists():
            try:
                cached = json.loads(path.read_text(encoding="utf-8"))["text"]
            except (OSError, ValueError):                         # half-written by a parallel call: treat as missing
                cached = None
        if cached is not None:
            text = cached
        elif OFFLINE:
            raise RuntimeError(f"OFFLINE and not in cache: {user[:80]!r}")
        else:
            try:
                reply = client.chat.completions.create(**payload)
                msg = reply.choices[0].message
                text = msg.content or getattr(msg, "reasoning_content", None) or ""   # some servers file a no-thinking reply under reasoning_content
            except Exception as e:                                # connection / server error: wait and retry
                time.sleep(3 * (attempt + 1))
                if attempt == retries - 1:
                    raise RuntimeError(f"endpoint failed {retries} times: {e}") from e
                continue
            tmp = path.with_name(path.name + f".{os.getpid()}.{id(payload)}.tmp")   # atomic write: parallel calls never see a half file
            tmp.write_text(json.dumps({"text": text, "payload": payload}, ensure_ascii=False), encoding="utf-8")
            os.replace(tmp, path)
        try:
            return _parse(text, schema)
        except (json.JSONDecodeError, ValidationError) as e:      # reply did not fit: drop it from the cache, ask again
            path.unlink(missing_ok=True)
            payload["temperature"] = 0.2
            if attempt == retries - 1:
                raise RuntimeError(f"reply did not match {schema.__name__}: {e}\n{text[:300]}") from e


def ask_many(calls, workers=6):                         # calls: list of dicts with ask()'s arguments
    with ThreadPoolExecutor(workers) as pool:
        return list(pool.map(lambda c: ask(**c), calls))


def need_file(name):
    """Make sure `name` exists in the session: on Colab, offer an upload dialog if it is missing."""
    if not Path(name).exists():
        try:
            from google.colab import files
            print(f"{name} not found - please upload it:")
            files.upload()
        except Exception:
            raise FileNotFoundError(f"{name} not found in the working directory")
    return name


In [ ]:
LEVEL_HINTS = {
    'B1': 'everyday vocabulary, mostly simple sentences with a few longer ones, concrete topics',
    'B2': 'wider vocabulary, longer sentences with subordinate clauses, some abstract ideas, a point of view that is not spelled out',
    'C1': 'precise and sometimes rare vocabulary, dense multi-clause sentences, an argument with evidence and an implicit stance',
}

PASSAGE_SYSTEM = """You write reading passages for English language tests."""

PASSAGE_PROMPT = """Write a reading passage in English for learners at CEFR level {{level}} ({{hint}}).
Topic: {{topic}}
Length: about {{n_words}} words, in 3 to 4 paragraphs.

The passage will be used for comprehension questions, so it must contain:
- several concrete facts (names, numbers, places, causes), each stated only once;
- something a reader can conclude that the text never says;
- information in different paragraphs that only makes full sense when combined;
- a recognisable point of view of the writer.

Do not open with a sentence that summarises the passage, and do not close with one that repeats it."""

CEFR_SYSTEM = """You are an examiner who knows the CEFR levels A1 to C2."""

CEFR_PROMPT = """Which CEFR level (A1, A2, B1, B2, C1 or C2) does a reader need for this text?
Judge the text as a whole: vocabulary, grammar and sentence length.

Text:
{{text}}"""

REWRITE_SYSTEM = """You adapt texts to CEFR levels without changing their content."""

REWRITE_PROMPT = """This text is at CEFR level {{measured}}. Rewrite it so that it fits CEFR level {{target}}.
Keep every fact, name and number, the paragraph structure and the writer's point of view.
Do not add explanations or a summary.

Text:
{{text}}"""

CATEGORIES = {
    'FR': {
        'name': 'Focus and Retrieve',
        'definition': 'The answer is stated in the text and the reader has to find it. Question and answer must be paraphrased, so that spotting a keyword is not enough.',
        'check': 'Does the question or the answer reuse the wording of the text? If yes, rephrase it.',
        'distractor': 'Each distractor must be something the text does NOT support anywhere.',
    },
    'INF': {
        'name': 'Inference',
        'definition': 'The answer is not stated in the text. The reader has to conclude it from clues the text gives.',
        'check': 'Could the answer be found by combining sentences of the text? If yes, it is not an inference - replace it.',
        'distractor': 'Each distractor must NOT be a valid conclusion from the text.',
    },
    'INT': {
        'name': 'Interpret and Integrate',
        'definition': 'The answer only follows from combining two different parts of the text. Neither part gives it alone.',
        'check': 'Could a reader who read only one of the two parts answer it? If yes, replace it.',
        'distractor': 'Each distractor is a wrong connection between the two parts, or a conclusion from one part only.',
    },
    'CRIT': {
        'name': 'Evaluate and Critique',
        'definition': 'The reader judges how the text is written: how effective an argument, an example, the structure or the ending is, and why.',
        'check': 'Is the answer a judgement about the writing rather than a summary of the content, and is the question not a yes/no question? If not, replace it.',
        'distractor': 'Each distractor must be a judgement the text clearly contradicts - not just a weaker judgement.',
    },
}

STEM_KEY_SYSTEM = """You write reading comprehension questions for English language tests."""

STEM_KEY_PROMPT = """Write {{n}} question(s) about the text, each with its correct answer only (no wrong options yet).

Category: {{name}}
{{definition}}
Before you output a question, check: {{check}}

Question: at most 150 characters. Answer: at most 90 characters. Language level: CEFR {{level}}.
Each question must target different information.

Text:
{{text}}"""

RATING_SYSTEM = """You are a strict reviewer of test questions."""

RATING_PROMPT = """Rate each question-answer pair from 1 to 10.

Category: {{name}}
{{definition}}

Criteria:
1. The pair fits the category (most important).
2. The question is clear, asks one thing only, contains no negation and no absolute words, and does not copy the text.
3. The answer is correct, short, and does not copy the text.

10 = no flaw; 7-9 = a minor flaw; 4-6 = one clear flaw; 1-3 = does not fit the category or is wrong.

Text:
{{text}}

Pairs:
{{pairs}}"""

FEEDBACK_SYSTEM = """You are an editor of test questions."""

FEEDBACK_PROMPT = """This question-answer pair was rated {{score}}/10.

Category: {{name}}
{{definition}}

Say what is wrong with the pair (category fit, clarity, copied wording, correctness) and suggest one concrete improvement.

Text:
{{text}}

Question: {{question}}
Answer: {{answer}}"""

IMPROVE_SYSTEM = """You are an editor of test questions."""

IMPROVE_PROMPT = """Improve this question-answer pair following the feedback. Keep the category ({{name}}).
Question: at most 150 characters. Answer: at most 90 characters.

Text:
{{text}}

Question: {{question}}
Answer: {{answer}}
Feedback: {{feedback}}"""

DISTRACTOR_SYSTEM = """You write multiple-choice options for reading tests."""

DISTRACTOR_PROMPT = """Write {{n}} wrong answer options (distractors) for this question.

Category: {{name}}. {{distractor}}
Every distractor must be clearly wrong for someone who read the text carefully, but tempting for someone who did not.
Match the correct answer in length and grammar. Do not write "all of the above" or "none of the above".
For each distractor, say briefly why it is wrong.

Text:
{{text}}

Question: {{question}}
Correct answer: {{answer}}"""

SELECT_SYSTEM = """You are a strict reviewer of multiple-choice options."""

SELECT_PROMPT = """Choose the {{k}} best distractors for this question from the candidates.
A good distractor is clearly wrong for a careful reader, tempting for a careless one, similar in length and grammar to the correct answer, does not overlap with the other chosen distractors, and does not give the answer away.
Copy the chosen distractors exactly as written.

Text:
{{text}}

Question: {{question}}
Correct answer: {{answer}}
Candidates:
{{candidates}}"""

EXPLAIN_SYSTEM = """You are a reading teacher."""

EXPLAIN_PROMPT = """Explain in two sentences why the correct answer is right. Quote the evidence from the text.

Text:
{{text}}

Question: {{question}}
Correct answer: {{answer}}"""

## Output schemas

Every call that has to return data gets a small pydantic class. The field names are what the
model has to produce; the descriptions are part of the instructions. Two helpers format items
for the prompts and for printing.

In [ ]:
class Passage(BaseModel):
    title: str
    text: str

class Level(BaseModel):
    reasoning: str = Field(description="One or two sentences on vocabulary, grammar and sentence length.")
    level: str = Field(description="One of A1, A2, B1, B2, C1, C2.")

class StemKey(BaseModel):
    question: str = Field(description="The question, at most 150 characters.")
    answer: str = Field(description="The correct answer, at most 90 characters.")

class StemKeys(BaseModel):
    items: list[StemKey]

class Rating(BaseModel):
    reasoning: str = Field(description="One sentence on the main strength or flaw.")
    score: int = Field(description="1 to 10.")

class Ratings(BaseModel):
    ratings: list[Rating] = Field(description="One rating per pair, in the order given.")

class Feedback(BaseModel):
    problem: str
    suggestion: str

class Distractor(BaseModel):
    text: str
    why_wrong: str = Field(description="One sentence: why this option is wrong according to the text.")

class Distractors(BaseModel):
    distractors: list[Distractor]

class Selected(BaseModel):
    distractors: list[str] = Field(description="The chosen distractors, copied exactly.")

class Explanation(BaseModel):
    explanation: str


def pairs_text(cands):                                  # numbered Q/A lines for the rating prompt
    return "\n".join(f"{i + 1}. Q: {c['question']}\n   A: {c['answer']}" for i, c in enumerate(cands))


def show(item):
    print(f"[{item['id']}] {item['question']}")
    for L, text in item["options"].items():
        print(f"   {'*' if L == item['correct'] else ' '} {L}) {text}")

## Settings

Five numbers drive the pipeline.

In [ ]:
N_CANDIDATES  = 3    # question/answer pairs per passage and category (3 passages x 4 categories x 3 = 36)
PASS_SCORE    = 7    # a pair needs this rating to be kept
N_ITEMS       = 10   # how many items to export
N_DISTRACTORS = 6    # distractors written per item before selecting the best 3
REFINE_ROUNDS = 1    # feedback -> rewrite -> re-rate rounds for weak pairs (at most 2)
SEED          = 1    # passed to the endpoint for the sampled steps, so that a rerun gives the same passages

## Step 1 · Write three passages

`passage_call(level, topic, n_words)` builds one call:
1. picks the one-line description of what the level looks like (`LEVEL_HINTS`);
2. fills the passage prompt — the prompt asks for facts, an unstated conclusion, information
   spread over paragraphs and a point of view, because the four categories need exactly that;
3. names the text model (`TEXT_MODEL`) and the `Passage` schema for the reply.

The three calls run in parallel with `ask_many`; each reply becomes a dict with an id.

In [ ]:
SPECS = [  # level, topic, words
    ("B1", "a volunteer repair café in a small town, and the local shop that objects to it", 260),
    ("B2", "an island community choosing between a bridge to the mainland and a new ferry", 360),
    ("C1", "an opinion piece arguing that 'quiet hours' in open-plan offices treat a symptom, not the cause", 450),
]


def passage_call(level, topic, n_words):
    prompt = render(PASSAGE_PROMPT, level=level, hint=LEVEL_HINTS[level], topic=topic, n_words=n_words)
    return dict(system=PASSAGE_SYSTEM, user=prompt, schema=Passage, model=TEXT_MODEL, temperature=0.7, seed=SEED)


replies = ask_many([passage_call(*spec) for spec in SPECS])

passages = []
for (level, topic, n_words), reply in zip(SPECS, replies):
    passages.append({"id": f"P_{level}", "level": level, "topic": topic, "title": reply.title, "text": reply.text})
    print(f"P_{level}: {reply.title!r}, {len(reply.text.split())} words")
by_id = {p["id"]: p for p in passages}

## Step 2 · The level gate

Models often write above the requested level. `check_level(text)` asks an examiner prompt for the
CEFR level of the text as a whole. `fix_level(passage, measured)` rewrites the passage to the
target level while keeping every fact and the point of view — that matters, because the items
will be built on those facts.

Rule: rewrite only if the measured level is **more than one step** away from the target;
B2 measured as C1 is accepted, B2 measured as C2 is rewritten.

In [ ]:
LEVELS = ["A1", "A2", "B1", "B2", "C1", "C2"]


def check_level(text):
    return ask(CEFR_SYSTEM, render(CEFR_PROMPT, text=text), schema=Level, temperature=0.0, think=False).level


def fix_level(passage, measured):
    prompt = render(REWRITE_PROMPT, measured=measured, target=passage["level"], text=passage["text"])
    return ask(REWRITE_SYSTEM, prompt, schema=Passage, model=TEXT_MODEL).text


for p in passages:
    measured = check_level(p["text"])
    off_by = abs(LEVELS.index(measured) - LEVELS.index(p["level"])) if measured in LEVELS else 0
    print(f"{p['id']}: target {p['level']}, measured {measured} ->", "rewrite" if off_by > 1 else "keep")
    if off_by > 1:
        p["text"] = fix_level(p, measured)
        print(f"   after rewrite: measured {check_level(p['text'])}, {len(p['text'].split())} words")

for p in passages:
    print("\n" + "=" * 90 + f"\n{p['id']} · {p['title']}\n\n{p['text']}")

## Step 3 · Question + answer candidates

`stem_key_call(passage, cat, n)` is the Notebook 1 function as one call: category definition and
check into the prompt, the `StemKeys` schema for the output. We ask for `N_CANDIDATES` pairs per
passage and category — more than we need, because the next steps will throw some away. Each
candidate becomes a dict that carries its passage, its category and later its ratings.

In [ ]:
def stem_key_call(passage, cat, n):
    c = CATEGORIES[cat]
    prompt = render(STEM_KEY_PROMPT, n=n, name=c["name"], definition=c["definition"], check=c["check"],
                    level=passage["level"], text=passage["text"])
    return dict(system=STEM_KEY_SYSTEM, user=prompt, schema=StemKeys)


jobs = [(p, cat) for p in passages for cat in CATEGORIES]
replies = ask_many([stem_key_call(p, cat, N_CANDIDATES) for p, cat in jobs])

candidates = []
for (p, cat), reply in zip(jobs, replies):
    for k, it in enumerate(reply.items, start=1):
        candidates.append({"id": f"{p['id']}-{cat}-{k}", "passage": p["id"], "cat": cat,
                           "question": it.question.strip(), "answer": it.answer.strip()})
print(len(candidates), "candidates")

## Step 3b · Two quick rules before spending any rating

Cheap checks come first. `telegraphs(q, a)` flags a pair whose question and answer share a
content word of four letters or more — a reader can then match words instead of reading.
`too_long()` applies the character limits from the prompt.

In [ ]:
STOPWORDS = set("about after again also because before being between could does from have into more most other "
                "over some such than that their there these they this those through under what when where which while "
                "with would your text passage writer author according".split())


def words(s):
    return {w for w in re.findall(r"[a-z']+", s.lower()) if len(w) >= 4 and w not in STOPWORDS}


def telegraphs(question, answer):
    return bool(words(question) & words(answer))


def too_long(c):
    return len(c["question"]) > 150 or len(c["answer"]) > 90


kept = []
for c in candidates:
    if telegraphs(c["question"], c["answer"]):
        print(f"dropped {c['id']} (question and answer share a word): {c['question']} -> {c['answer']}")
    elif too_long(c):
        print(f"dropped {c['id']} (too long)")
    else:
        kept.append(c)
candidates = kept
print(len(candidates), "candidates left")

## Step 4 · Rating

`rate(cands)` groups the candidates by passage and category (one call per group, so the rater
sees siblings side by side), fills the rating prompt with the category definition and the pairs,
and writes a `score` and a one-sentence `reasoning` into every candidate dict. If the model
returns the wrong number of ratings — models do miscount lists — the group is asked once more.

The rubric is short: category fit first, then clarity and copied wording, then correctness of the
answer. 10 means no flaw.

In [ ]:
def rate(cands):
    groups = {}
    for c in cands:
        groups.setdefault((c["passage"], c["cat"]), []).append(c)
    calls = []
    for (pid, cat), grp in groups.items():
        c = CATEGORIES[cat]
        prompt = render(RATING_PROMPT, name=c["name"], definition=c["definition"], text=by_id[pid]["text"], pairs=pairs_text(grp))
        calls.append(dict(system=RATING_SYSTEM, user=prompt, schema=Ratings, temperature=0.2))
    replies = ask_many(calls)
    for grp, call, reply in zip(groups.values(), calls, replies):
        if len(reply.ratings) != len(grp):                       # models miscount lists: ask once more, explicitly
            reply = ask(call["system"], call["user"] + f"\n\nReturn exactly {len(grp)} ratings, one per pair.", schema=Ratings, temperature=0.2)
            if len(reply.ratings) != len(grp):
                raise ValueError(f"rating returned {len(reply.ratings)} ratings for {len(grp)} pairs")
        for cand, r in zip(grp, reply.ratings):
            cand["score"], cand["reasoning"] = r.score, r.reasoning
    return cands


rate(candidates)
pd.DataFrame(candidates)[["id", "score", "question", "answer"]]

## Step 5 · Refinement (feedback → rewrite → re-rate)

For every candidate below `PASS_SCORE`:
1. `feedback(c)` asks an editor what is wrong and for one concrete improvement;
2. `improve(c, fb)` asks for a rewritten pair that follows the feedback;
3. the rewritten pairs are rated again with `rate()`;
4. the new version replaces the old one only if it scored higher.

`REFINE_ROUNDS` repeats this, at most twice — after that, rewriting rarely helps.

In [ ]:
def feedback_call(c):
    cat = CATEGORIES[c["cat"]]
    prompt = render(FEEDBACK_PROMPT, score=c["score"], name=cat["name"], definition=cat["definition"],
                    text=by_id[c["passage"]]["text"], question=c["question"], answer=c["answer"])
    return dict(system=FEEDBACK_SYSTEM, user=prompt, schema=Feedback, temperature=0.2)


def improve_call(c, fb):
    cat = CATEGORIES[c["cat"]]
    prompt = render(IMPROVE_PROMPT, name=cat["name"], text=by_id[c["passage"]]["text"], question=c["question"],
                    answer=c["answer"], feedback=f"{fb.problem} Suggestion: {fb.suggestion}")
    return dict(system=IMPROVE_SYSTEM, user=prompt, schema=StemKey, temperature=0.2)


def refine(cands):
    weak = [c for c in cands if c["score"] < PASS_SCORE]
    if not weak:
        print("nothing to refine")
        return cands
    feedbacks = ask_many([feedback_call(c) for c in weak])                       # 1
    rewrites = ask_many([improve_call(c, fb) for c, fb in zip(weak, feedbacks)])  # 2
    new = []
    for c, fb, rw in zip(weak, feedbacks, rewrites):
        v = {**c, "question": rw.question.strip(), "answer": rw.answer.strip(), "feedback": fb.problem}
        if not telegraphs(v["question"], v["answer"]) and not too_long(v):
            new.append(v)
    rate(new)                                                                    # 3
    better = {v["id"]: v for v in new if v["score"] > next(c["score"] for c in cands if c["id"] == v["id"])}
    print(f"{len(weak)} weak pairs, {len(better)} improved")
    return [better.get(c["id"], c) for c in cands]                               # 4


for _ in range(min(REFINE_ROUNDS, 2)):
    candidates = refine(candidates)
pd.DataFrame(candidates)[["id", "score", "question", "answer"]]

## Step 6 · Keep the good ones, pick ten

Candidates at or above `PASS_SCORE` are accepted. To export a balanced set, `pick()` walks
round-robin through the passage × category groups, best-rated first, until `N_ITEMS` are chosen.

In [ ]:
accepted = [c for c in candidates if c["score"] >= PASS_SCORE]
print(f"{len(accepted)} of {len(candidates)} accepted at score >= {PASS_SCORE}")


def pick(cands, n):
    groups = {}
    for c in sorted(cands, key=lambda c: -c["score"]):
        groups.setdefault((c["passage"], c["cat"]), []).append(c)
    chosen = []
    while len(chosen) < n and any(groups.values()):
        for grp in groups.values():
            if grp and len(chosen) < n:
                chosen.append(grp.pop(0))
    return chosen


selected = pick(accepted, N_ITEMS)
pd.DataFrame(selected)[["id", "score", "question", "answer"]]

## Step 7 · Distractors in two phases

1. `distractor_call(c, n)` — the Notebook 1 prompt: `N_DISTRACTORS` candidates with the
   category's distractor rule and a "why wrong" sentence each.
2. `gate(distractors, answer)` — a rule-based filter: no "all/none of the above", no vague words
   like "sometimes", and no distractor much longer than the correct answer (a much longer option
   is a known giveaway). If fewer than three survive, the rejected ones are added back in order,
   so that the reviewer in step 3 still has something to choose from.
3. `select_call(c, k)` — a reviewer prompt chooses the `k` best from what survived, copying
   them verbatim; anything it invents is ignored.

Writing more than needed and then selecting is cheaper than writing three perfect ones.

In [ ]:
VAGUE = ["sometimes", "often", "usually", "rarely", "generally", "somewhat", "partly", "occasionally"]


def distractor_call(c, n):
    cat = CATEGORIES[c["cat"]]
    prompt = render(DISTRACTOR_PROMPT, n=n, name=cat["name"], distractor=cat["distractor"],
                    text=by_id[c["passage"]]["text"], question=c["question"], answer=c["answer"])
    return dict(system=DISTRACTOR_SYSTEM, user=prompt, schema=Distractors, temperature=0.6, seed=SEED)


def gate(distractors, answer, needed=3):
    distractors = [d for d in dict.fromkeys(distractors) if d != answer]      # no duplicates, never the key
    ok, rejected = [], []
    for d in distractors:
        low = d.lower()
        too_long = len(d) > 1.5 * len(answer) + 10                            # allowance for very short keys
        (rejected if "of the above" in low or any(v in low.split() for v in VAGUE) or too_long else ok).append(d)
    return ok + rejected[: max(0, needed - len(ok))]                          # top up if the gate was too strict


def select_call(c, k):
    prompt = render(SELECT_PROMPT, k=k, text=by_id[c["passage"]]["text"], question=c["question"], answer=c["answer"],
                    candidates="\n".join(f"- {d}" for d in c["pool"]))
    return dict(system=SELECT_SYSTEM, user=prompt, schema=Selected, temperature=0.2)


replies = ask_many([distractor_call(c, N_DISTRACTORS) for c in selected])          # phase 1
for c, reply in zip(selected, replies):
    c["pool"] = gate([d.text for d in reply.distractors], c["answer"])            # gate
    print(f"{c['id']}: {len(reply.distractors)} written, {len(c['pool'])} after the gate")

replies = ask_many([select_call(c, 3) for c in selected])                           # phase 2
for c, reply in zip(selected, replies):
    chosen = [d for d in reply.distractors if d in c["pool"]]
    c["distractors"] = (chosen + [d for d in c["pool"] if d not in chosen])[:3]

## Step 8 · Assemble, explain, export

`assemble(c)` sorts key and distractors alphabetically into A–D and records the key's letter.
`explain(item)` asks a teacher prompt for a two-sentence explanation with the evidence from the
text. The export is one JSON file with the passages and the items — Notebook 3 reads it.

The item format is deliberately small:
`id, passage, cat, level, question, options {A..D}, correct, explanation, score`.

In [ ]:
def assemble(c):
    texts = sorted([c["answer"]] + c["distractors"], key=str.lower)
    options = {L: t for L, t in zip("ABCD", texts)}
    return {"id": c["id"], "passage": c["passage"], "cat": c["cat"], "level": by_id[c["passage"]]["level"],
            "question": c["question"], "options": options,
            "correct": [L for L, t in options.items() if t == c["answer"]][0], "score": c["score"]}


def explain_call(item):
    prompt = render(EXPLAIN_PROMPT, text=by_id[item["passage"]]["text"], question=item["question"],
                    answer=item["options"][item["correct"]])
    return dict(system=EXPLAIN_SYSTEM, user=prompt, schema=Explanation, temperature=0.2)


items = [assemble(c) for c in selected if len(c["distractors"]) == 3]
for item, reply in zip(items, ask_many([explain_call(it) for it in items])):
    item["explanation"] = reply.explanation

for item in items:
    show(item)
    print("   ->", item["explanation"], "\n")

## Math items from the hand-written template

One Gierl-style item model, written by hand: a two-step percentage problem. The template fixes
the numbers, the solution path, and the misconception behind every distractor; nothing numerical
comes from a model.

* **Elements:** `P` (original price) and `r` (rate of decrease); `Q = P · (1 − r)`.
* **Key:** `100 · r`.
* **Distractors, one misconception each:** the decrease taken relative to the *new* price
  (`100 · (P − Q) / Q`, "wrong base"); the absolute difference `P − Q` read as a percentage; the
  remaining fraction `100 · Q / P`.
* **Constraints:** whole-number decrease (`P · r` is an integer, otherwise rounding `Q` would make the key wrong), integer key, all four values distinct and positive.
* **Radicals:** `r` and the base; **incidentals:** the object, the currency, the shop.

`percent_decrease_item()` instantiates the model once; the loop below walks through all 5 × 4
combinations (the isomorph family; one is dropped by the constraints) and keeps `N_MATH` of them. Option order is shuffled with a
fixed seed. If `REALISE_WITH_LLM` is on, the text model rewrites each stem into a short, natural
context — the numbers are checked afterwards and the template stem is kept if any number changed.


In [ ]:
import random

N_MATH           = 10
REALISE_WITH_LLM = True     # let the text model phrase the stem; numbers are verified afterwards

CONTEXTS = [("a jacket", "€"), ("a bicycle", "€"), ("a pair of headphones", "€"), ("a train ticket", "CHF"),
            ("a textbook", "€"), ("a concert ticket", "£"), ("a lamp", "€"), ("a backpack", "CHF")]
P_VALUES = [40, 50, 60, 80, 120]           # original price
R_VALUES = [0.10, 0.20, 0.25, 0.40]        # rate of decrease


def percent_decrease_item(k, P, r, context, rng):
    thing, cur = context
    if abs(P * r - round(P * r)) > 1e-9:                       # constraint: the decrease must be a whole amount,
        return None                                            #   otherwise rounding Q would make the key wrong
    Q, key = round(P * (1 - r)), round(100 * r)
    values = {key: "key",
              round(100 * (P - Q) / Q): "wrong base",          # decrease relative to the new price
              P - Q: "absolute difference",                    # euros read as percent
              round(100 * Q / P): "remaining fraction"}        # what is left, not what was taken
    if len(values) < 4 or min(values) <= 0:                    # constraint: four distinct positive values
        return None
    order = list(values.items()); rng.shuffle(order)
    options = {L: f"{v} %" for L, (v, _) in zip("ABCD", order)}
    return {"id": f"M-{k:02d}", "passage": None, "cat": "MATH", "level": None,
            "question": f"A shop reduces the price of {thing} from {cur}{P} to {cur}{Q}. By what percentage has the price decreased?",
            "options": options, "correct": [L for L, (v, _) in zip("ABCD", order) if v == key][0],
            "explanation": f"Decrease = {P} - {Q} = {P - Q}; {P - Q} / {P} = {r:.2f} = {key} %.", "score": None,
            "template": {"P": P, "Q": Q, "r": r, "misconceptions": {L: name for L, (v, name) in zip("ABCD", order)}}}


rng = random.Random(SEED)
combos = [(P, r) for P in P_VALUES for r in R_VALUES]; rng.shuffle(combos)        # the 20 isomorphs, in random order
MATH_ITEMS = [it for it in (percent_decrease_item(k, P, r, CONTEXTS[(k - 1) % len(CONTEXTS)], rng)
                            for k, (P, r) in enumerate(combos, start=1)) if it][:N_MATH]


class MathStem(BaseModel):
    question: str = Field(description="The rewritten question, one or two sentences, all numbers unchanged.")


REALISE_PROMPT = """Rewrite this maths question as a short, natural word problem for 13-year-olds. Keep every number and the currency exactly as they are, keep it a question about the percentage decrease, and do not add hints or the answer.

{{question}}"""

if REALISE_WITH_LLM:
    replies = ask_many([dict(system="You write maths word problems for school tests.", user=render(REALISE_PROMPT, question=it["question"]),
                             schema=MathStem, model=TEXT_MODEL, temperature=0.7, seed=SEED) for it in MATH_ITEMS])
    for it, reply in zip(MATH_ITEMS, replies):
        t = it["template"]
        if str(t["P"]) in reply.question and str(t["Q"]) in reply.question:     # numbers survived: use the model's wording
            it["template_stem"], it["question"] = it["question"], reply.question.strip()

for it in MATH_ITEMS:
    show(it)
    print("   misconceptions:", {L: n for L, n in it["template"]["misconceptions"].items() if n != "key"}, "\n")

In [ ]:
with open("day1_items.json", "w", encoding="utf-8") as f:
    json.dump({"passages": passages, "items": items + MATH_ITEMS}, f, indent=2, ensure_ascii=False)
print("reading items:", len(items), " math items:", len(MATH_ITEMS))
print(f"saved {len(items) + len(MATH_ITEMS)} items and {len(passages)} passages to day1_items.json")